# Cross-Industry Accelerator — Load Warehouse
### Auto-Load Tables into Fabric Data Warehouse with DDL Generation

This notebook loads **all discovered tables** into the Fabric Data Warehouse,
dynamically generating `CREATE TABLE` DDL from inferred Spark schemas.

**What it does:**
1. Reads industry configuration from `00_Industry_Config`
2. For each table (dim, fact-batch, fact-event, stream):
   - Infers schema from CSV
   - Generates SQL `CREATE TABLE IF NOT EXISTS` DDL
   - Loads data via `synapsesql()` connector
3. Generates a load summary with row counts and status

> **Prerequisites:**
> 1. Run `01_Data_Ingestion.ipynb` to validate sources
> 2. A Fabric **Data Warehouse** must exist (name configured in `00_Industry_Config`)
> 3. The notebook must have connectivity to the warehouse endpoint

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RUN CONFIGURATION NOTEBOOK
# ════════════════════════════════════════════════════════════════════════

In [ ]:
%run 00_Industry_Config

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# TYPE MAPPING & DDL GENERATION UTILITIES (ZT-hardened)
# ════════════════════════════════════════════════════════════════════════

from pyspark.sql.types import (
    StringType, IntegerType, LongType, FloatType, DoubleType,
    BooleanType, DateType, TimestampType, DecimalType, ShortType, ByteType
)
import pandas as pd

# Spark type → Warehouse SQL type mapping (Fabric Warehouse edition)
SPARK_TO_SQL = {
    StringType:    "VARCHAR(8000)",
    IntegerType:   "INT",
    LongType:      "BIGINT",
    ShortType:     "SMALLINT",
    ByteType:      "TINYINT",
    FloatType:     "FLOAT",
    DoubleType:    "FLOAT",
    BooleanType:   "BIT",
    DateType:      "DATE",
    TimestampType: "DATETIME2(6)",
}

def spark_type_to_sql(spark_type):
    """Convert a Spark DataType instance to SQL Server type string."""
    if isinstance(spark_type, DecimalType):
        return f"DECIMAL({spark_type.precision}, {spark_type.scale})"
    return SPARK_TO_SQL.get(type(spark_type), "VARCHAR(8000)")

def generate_ddl(table_name, schema, warehouse_schema="dbo"):
    """Generate CREATE TABLE IF NOT EXISTS DDL from Spark schema.
    ZT: All identifiers are validated before interpolation into SQL."""
    # ZT: Validate table name and schema name
    sanitize_identifier(table_name)
    sanitize_identifier(warehouse_schema)

    # ZT: Validate and build column definitions with sanitized names
    col_defs = []
    for field in schema.fields:
        sanitize_identifier(field.name)  # Raises ValueError if unsafe
        null_clause = "" if field.nullable else " NOT NULL"
        col_defs.append(f"[{field.name}] {spark_type_to_sql(field.dataType)}{null_clause}")

    columns = ",\n    ".join(col_defs)
    return f"""IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = '{table_name}')
CREATE TABLE [{warehouse_schema}].[{table_name}] (
    {columns}
)"""

print("✅ DDL generation utilities loaded (ZT: column name sanitization enabled).")
print(f"   Warehouse target: {WAREHOUSE_NAME}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# LOAD ALL TABLES → FABRIC DATA WAREHOUSE (via pyodbc + SQL endpoint)
# ════════════════════════════════════════════════════════════════════════
# Auto-creates the Warehouse if it doesn't exist.
# Uses pyodbc with AAD token auth to write directly to the Warehouse.

import requests as req_lib
import pyodbc
import struct
import time
import sempy.fabric as fabric

ALL_TABLES = DIM_TABLES + FACT_TABLES_BATCH + FACT_TABLES_EVENT + STREAM_TABLES
warehouse_schema = "dbo"
results = []

# ── Get workspace context ────────────────────────────────────────────
workspace_id = fabric.get_workspace_id()
pbi_token = notebookutils.credentials.getToken('pbi')
api_headers = {
    "Authorization": f"Bearer {pbi_token}",
    "Content-Type": "application/json",
}
BASE_URL = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"

# ── Find or create the Warehouse (with retries for transient errors) ─
def _list_items_with_retry(max_retries=3):
    for attempt in range(max_retries):
        try:
            return fabric.list_items()
        except Exception as e:
            if attempt < max_retries - 1:
                print(f"  Fabric API error (attempt {attempt+1}), retrying in 10s...")
                time.sleep(10)
            else:
                raise

items_df = _list_items_with_retry()
# Fabric API returns "Warehouse" but sempy may return "DataWarehouse" — check both
wh_match = items_df[(items_df["Type"].isin(["DataWarehouse", "Warehouse"])) & (items_df["Display Name"] == WAREHOUSE_NAME)]

if wh_match.empty:
    print(f"Warehouse '{WAREHOUSE_NAME}' not found. Creating...")
    body = {"displayName": WAREHOUSE_NAME, "type": "Warehouse"}
    resp = req_lib.post(f"{BASE_URL}/items", headers=api_headers, json=body)

    if resp.status_code == 202:
        op_url = resp.headers.get("Location", "")
        for _ in range(30):
            time.sleep(5)
            poll = req_lib.get(op_url, headers=api_headers)
            if poll.status_code == 200 and poll.json().get("status") == "Succeeded":
                break
    elif resp.status_code not in (200, 201):
        raise RuntimeError(f"Failed to create Warehouse: {resp.status_code} {resp.text[:200]}")

    time.sleep(10)  # Wait for provisioning
    # Poll until warehouse appears (up to 3 minutes)
    for wait in range(18):
        items_df = _list_items_with_retry()
        wh_match = items_df[(items_df["Type"].isin(["DataWarehouse", "Warehouse"])) & (items_df["Display Name"] == WAREHOUSE_NAME)]
        if not wh_match.empty:
            break
        print(f"  Waiting for provisioning... ({(wait+1)*10}s)")
        time.sleep(10)
    if wh_match.empty:
        raise RuntimeError(f"Warehouse '{WAREHOUSE_NAME}' still provisioning after 3 min. Check Fabric portal.")
    print(f"  Warehouse '{WAREHOUSE_NAME}' created -> {wh_match.iloc[0].Id}")
else:
    print(f"  Warehouse '{WAREHOUSE_NAME}' exists -> {wh_match.iloc[0].Id}")

wh_id = wh_match.iloc[0].Id

# ── Get SQL endpoint ────────────────────────────────────────────────
wh_resp = req_lib.get(f"{BASE_URL}/warehouses/{wh_id}", headers=api_headers)
wh_props = wh_resp.json().get("properties", {})
sql_endpoint = wh_props.get("connectionString", "")

if not sql_endpoint:
    print(f"Could not get SQL endpoint. Full response:")
    print(wh_resp.json())
    raise RuntimeError("No connectionString in warehouse properties")

print(f"  SQL Endpoint: {sql_endpoint}")
print(f"  Tables: {len(ALL_TABLES)}\n")

# ── Connect to Warehouse via pyodbc with AAD token ──────────────────
sql_token = notebookutils.credentials.getToken("https://database.windows.net/")
token_bytes = sql_token.encode("UTF-16-LE")
token_struct = struct.pack(f'<I{len(token_bytes)}s', len(token_bytes), token_bytes)
SQL_COPT_SS_ACCESS_TOKEN = 1256

conn = pyodbc.connect(
    f"DRIVER={{ODBC Driver 18 for SQL Server}};"
    f"SERVER={sql_endpoint};"
    f"DATABASE={WAREHOUSE_NAME};"
    f"encrypt=yes;TrustServerCertificate=no",
    attrs_before={SQL_COPT_SS_ACCESS_TOKEN: token_struct},
)
conn.autocommit = True
cursor = conn.cursor()
print("Connected to Warehouse SQL endpoint.\n")

for table_name in ALL_TABLES:
    csv_path = f"{_FS_CSV_PATH}/{table_name}.csv"

    if table_name in DIM_TABLES:
        category = "Dimension"
    elif table_name in FACT_TABLES_BATCH:
        category = "Fact (Batch)"
    elif table_name in FACT_TABLES_EVENT:
        category = "Fact (Event)"
    else:
        category = "Streaming"

    try:
        sanitize_identifier(table_name)
    except ValueError as ve:
        results.append((table_name, category, 0, 0, f"ZT: {ve}"))
        print(f"  ZT BLOCKED {table_name}")
        continue

    try:
        df = (spark.read
              .option("header", True)
              .option("inferSchema", True)
              .csv(csv_path))
        pdf = df.toPandas()

        cursor.execute(f"DROP TABLE IF EXISTS [{warehouse_schema}].[{table_name}]")

        col_defs = []
        for field in df.schema.fields:
            sanitize_identifier(field.name)
            sql_type = spark_type_to_sql(field.dataType)
            col_defs.append(f"[{field.name}] {sql_type}")
        create_sql = f"CREATE TABLE [{warehouse_schema}].[{table_name}] ({', '.join(col_defs)})"
        cursor.execute(create_sql)

        if len(pdf) > 0:
            cols = [f"[{c}]" for c in pdf.columns]
            placeholders = ",".join(["?" for _ in cols])
            insert_sql = f"INSERT INTO [{warehouse_schema}].[{table_name}] ({','.join(cols)}) VALUES ({placeholders})"

            batch_size = 100
            for i in range(0, len(pdf), batch_size):
                batch = pdf.iloc[i:i+batch_size]
                rows = [tuple(None if pd.isna(v) else v for v in row) for row in batch.itertuples(index=False)]
                cursor.executemany(insert_sql, rows)

        row_count = len(pdf)
        results.append((table_name, category, row_count, len(pdf.columns), "ok"))
        log_audit_event("WAREHOUSE_LOAD", f"{WAREHOUSE_NAME}.{warehouse_schema}.{table_name}", f"{row_count} rows")
        print(f"  ok {table_name:<45} {row_count:>6} rows  {len(pdf.columns):>3} cols")

    except Exception as e:
        results.append((table_name, category, 0, 0, f"ERR: {e}"))
        log_audit_event("WAREHOUSE_LOAD", table_name, str(e), "ERROR")
        print(f"  ERR {table_name:<44} {str(e)[:80]}")

cursor.close()
conn.close()

ok_count = sum(1 for r in results if r[4] == 'ok')
print(f"\nTables loaded: {ok_count}/{len(ALL_TABLES)}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# GENERATED DDL PREVIEW
# ════════════════════════════════════════════════════════════════════════
# Re-read one table to show the DDL that was generated (for verification)

if ALL_TABLES:
    sample_table = ALL_TABLES[0]
    sample_path = f"{_FS_CSV_PATH}/{sample_table}.csv"
    sample_df = (spark.read
                 .option("header", True)
                 .option("inferSchema", True)
                 .csv(sample_path))
    print(f"Sample DDL for '{sample_table}':\n")
    print(generate_ddl(sample_table, sample_df.schema, warehouse_schema))

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# WAREHOUSE LOAD SUMMARY
# ════════════════════════════════════════════════════════════════════════

summary_df = pd.DataFrame(results, columns=["Table", "Category", "Rows", "Columns", "Status"])

print(f"\n{'═'*75}")
print(f"WAREHOUSE LOAD SUMMARY — {INDUSTRY.upper()}")
print(f"{'═'*75}")
print(summary_df.to_string(index=False))
print(f"\n{'─'*75}")

for cat in ["Dimension", "Fact (Batch)", "Fact (Event)", "Streaming"]:
    cat_df = summary_df[summary_df['Category'] == cat]
    if not cat_df.empty:
        ok = (cat_df['Status'] == 'ok').sum()
        total_rows = cat_df[cat_df['Status'] == 'ok']['Rows'].sum()
        print(f"  {cat:<16} {ok}/{len(cat_df)} tables  {total_rows:>8,} rows")

print(f"\n  TOTAL: {summary_df[summary_df['Status']=='ok']['Rows'].sum():,} rows loaded")
print(f"\n✅ Warehouse loading complete.")
print(f"   Next: Run 04_Create_Semantic_Model.ipynb to create the Power BI semantic model.")